# 20. pgvector 실습 데이터 — 합성 벡터 10만 개와 정답지

2부(21~24)는 인덱스의 **동작**을 재는 실습이라 의미가 없는 합성 벡터를 쓴다.
다운로드가 없고, 성질(군집이 있나 없나)을 마음대로 바꿀 수 있기 때문이다.

| 테이블 | 내용 |
|---|---|
| `vec_clu` | **군집형**: 중심 200개 + 잡음, 128차원 10만 개 (실제 임베딩처럼 뭉쳐 있다) |
| `vec_iso` | **무구조형**: 128차원 등방 가우시안 10만 개 (군집이 전혀 없다) |
| `vec_query` | 두 데이터 각각의 질의 200개 (테이블에 없는 새 점) |
| `vec_gt` | **정답지**: 질의마다 정확한 최근접 100개 (전수 비교) |

모든 ANN 실험의 recall 은 이 정답지와 비교한 **ANN recall** 이다.

In [ ]:
from labutil import *

N, D, NQ = 100_000, 128, 200
conn = connect()

Xc, clu = make_clusters(N + NQ, D, n_clusters=200, center_scale=1.0, seed=42)
rng = np.random.default_rng(43)
Xi = rng.normal(0, 1, (N + NQ, D)).astype(np.float32)
tags = rng.integers(0, 100, N)            # 필터 실습(24번)용 속성: 벡터와 무관한 0~99

data = {"clu": (Xc[:N], Xc[N:], clu[:N]), "iso": (Xi[:N], Xi[N:], np.zeros(N, int))}

## 1. 테이블 만들고 COPY (binary)

In [ ]:
conn.execute("DROP TABLE IF EXISTS vec_clu, vec_iso, vec_query, vec_gt CASCADE")
for ds in data:
    conn.execute(f"CREATE TABLE vec_{ds} (id int PRIMARY KEY, cluster int, tag int, emb vector({D}))")
conn.execute(f"CREATE TABLE vec_query (ds text, qid int, emb vector({D}), PRIMARY KEY (ds, qid))")
conn.execute("CREATE TABLE vec_gt (ds text, qid int, rank int, id int, dist float4, PRIMARY KEY (ds, qid, rank))")

for ds, (X, Q, lab) in data.items():
    with timer(f"vec_{ds} 적재 {N:,}행"), conn.cursor() as cur:
        with cur.copy(f"COPY vec_{ds} (id, cluster, tag, emb) FROM STDIN WITH (FORMAT BINARY)") as cp:
            cp.set_types(["int4", "int4", "int4", "vector"])
            for i in range(N):
                cp.write_row((i, int(lab[i]), int(tags[i]), X[i]))
        with cur.copy("COPY vec_query (ds, qid, emb) FROM STDIN WITH (FORMAT BINARY)") as cp:
            cp.set_types(["text", "int4", "vector"])
            for j in range(NQ):
                cp.write_row((ds, j, Q[j]))
    conn.execute(f"ANALYZE vec_{ds}")

## 2. 정답지: 질의 200개 × 정확한 top-100
numpy 로 전수 비교한다 (200 × 10만 × 128 행렬곱이라 몇 초면 끝).

In [ ]:
for ds, (X, Q, _) in data.items():
    idx = brute_knn(X, Q, 100)
    with conn.cursor() as cur, cur.copy("COPY vec_gt (ds, qid, rank, id, dist) FROM STDIN") as cp:
        for j in range(NQ):
            dd = np.linalg.norm(X[idx[j]] - Q[j], axis=1)
            for r, (i, d_) in enumerate(zip(idx[j], dd), 1):
                cp.write_row((ds, j, r, int(i), float(d_)))
q(conn, "SELECT ds, count(DISTINCT qid) 질의, count(*) 정답행 FROM vec_gt GROUP BY ds")

PG 안에서 인덱스 없이 정렬하면(= 전수 비교) 정답지와 같은지 확인한다.

In [ ]:
qv = q(conn, "SELECT emb FROM vec_query WHERE ds='clu' AND qid=0")["emb"][0]
pg_ids = q(conn, "SELECT id FROM vec_clu ORDER BY emb <-> %s LIMIT 10", (qv,))["id"].tolist()
gt_ids = q(conn, "SELECT id FROM vec_gt WHERE ds='clu' AND qid=0 AND rank<=10 ORDER BY rank")["id"].tolist()
print("PG 전수 정렬:", pg_ids, "\n정답지     :", gt_ids, "\n일치:", pg_ids == gt_ids)
print(explain(conn, "SELECT id FROM vec_clu ORDER BY emb <-> %s LIMIT 10", (qv,)))

인덱스가 없으니 **Seq Scan → 10만 행 전부 거리 계산 → 정렬**. 이 속도가 21번부터의 기준선이다.

## 3. 두 데이터는 어떻게 다른가

In [ ]:
def pca2(X):
    """주성분 2개로 투영 (고차원 점을 평면에 그리기 위한 것)"""
    S = X - X.mean(0)
    _, _, Vt = np.linalg.svd(S, full_matrices=False)
    return S @ Vt[:2].T


fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
# 군집 200개를 한꺼번에 2차원에 그리면 뭉개진다 → 군집 8개만 골라 투영
pick = np.isin(data["clu"][2], np.arange(8))
P = pca2(data["clu"][0][pick])
axes[0].scatter(*P.T, s=3, c=data["clu"][2][pick], cmap="tab10", alpha=0.6)
axes[0].set_title("군집형 vec_clu: 군집 8개만 골라 2차원 투영")
P = pca2(data["iso"][0][:int(pick.sum())])
axes[1].scatter(*P.T, s=3, color="#3D484A", alpha=0.5)
axes[1].set_title("무구조형 vec_iso: 같은 개수를 투영")
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])

for ds, color in [("clu", "#FF7043"), ("iso", "#3D484A")]:
    d = q(conn, "SELECT rank, dist FROM vec_gt WHERE ds=%s", (ds,))
    X, Q, _ = data[ds]
    rnd = np.linalg.norm(X[np.random.default_rng(1).integers(0, N, 20000)] - np.repeat(Q[:100], 200, 0), axis=1)
    nn1 = d[d["rank"] == 1]["dist"]
    axes[2].hist(rnd / rnd.mean(), bins=60, alpha=0.35, color=color, density=True, label=f"{ds}: 아무 점까지")
    axes[2].axvline(nn1.mean() / rnd.mean(), color=color, lw=2, ls="--", label=f"{ds}: 최근접까지 (평균)")
axes[2].set_xlabel("거리 / 평균 거리"); axes[2].legend(fontsize=8); axes[2].set_title("최근접이 '특별히' 가까운가?")
plt.tight_layout(); plt.show()

**관찰:**
- 군집형은 최근접이 평균 거리의 60% 정도로 가깝다. 무구조형은 **최근접조차 평균의 77%** — 고차원에서 모든 점이 비슷하게 먼 현상(거리 집중)
- 무구조형에서는 "가까운 쪽으로 따라가는" 그래프도, "가까운 군집만 뒤지는" IVF 도 단서가 약하다 → 22번에서 비교

## 4. 크기

In [ ]:
sizes = q(conn, """
SELECT relname AS "테이블", pg_relation_size(oid) / 2^20 AS "본체_MB"
FROM pg_class WHERE relname IN ('vec_clu', 'vec_iso')""")
raw = N * D * 4 / 2**20
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(["float 원본 (N×D×4B)"] + sizes["테이블"].tolist(), [raw] + sizes["본체_MB"].tolist(),
        color=["#9aa5a8", "#FF7043", "#3D484A"])
for i, v in enumerate([raw] + sizes["본체_MB"].tolist()):
    ax.text(v, i, f" {v:.0f} MB", va="center")
ax.set_xlabel("MB"); ax.set_title("vector(128) 한 행 = 4B × 128 + 헤더"); plt.tight_layout(); plt.show()